In [1]:
import pandas as pd
import numpy as np

test = pd.Series([-1, -2, -3, -4, -5, np.nan, np.nan, np.nan, -6, -7, -8, -9, 0])

def custom_func(x):
    # Ignore NaN values
    valid_values = x[~np.isnan(x)]
    if len(valid_values) == 0:
        return np.nan  # Return NaN if no valid values
    last_value = valid_values.iloc[-1]
    return np.sum(valid_values < last_value) / len(valid_values)

# Set a smaller window size, e.g., 5 or 6
result = test.rolling(window=6).dropna().apply(custom_func, raw=False)
print(result)


0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
dtype: float64


In [2]:
ser = pd.Series([1,2,3,np.nan, 4,5,6,7,8])
ser

0    1.0
1    2.0
2    3.0
3    NaN
4    4.0
5    5.0
6    6.0
7    7.0
8    8.0
dtype: float64

In [7]:
ser.index

RangeIndex(start=0, stop=9, step=1)

method1

In [24]:
ser_nona = ser_nona.mask(ser_nona<6).dropna()
ser_nona

6    6.0
7    7.0
8    8.0
dtype: float64

method2 memory efficient

In [26]:
# Example Series
ser_nona = pd.Series([5, 6, 7, 8, 9, None, 10])

# Mask values and drop NaN values with inplace=True
ser_nona.mask(ser_nona < 6, inplace=True)  # Mask values less than 6
ser_nona.dropna(inplace=True)  # Drop NaN values

print(ser_nona)


1     6.0
2     7.0
3     8.0
4     9.0
6    10.0
dtype: float64


In [28]:
test2 = pd.Series([-1, -2, -3, -4, -5, 9, 1, np.nan, -6, -7, -8, -9, 0])
result2 = test2.rolling(window=6).apply(custom_func, raw=False)
print(result2)

0          NaN
1          NaN
2          NaN
3          NaN
4          NaN
5     0.833333
6     0.666667
7          NaN
8          NaN
9          NaN
10         NaN
11         NaN
12         NaN
dtype: float64


In [34]:
pd.isna(result2.iloc[-1])

True

In [52]:
df = pd.DataFrame(index= result2.index)
df['newC'] = pd.NA
df

,newC
0,<NA>
1,<NA>
2,<NA>
3,<NA>
4,<NA>
5,<NA>
6,<NA>
7,<NA>
8,<NA>
9,<NA>


In [55]:
df['b'] = pd.NA

In [56]:
df

,newC,b
0,<NA>,<NA>
1,<NA>,<NA>
2,<NA>,<NA>
3,<NA>,<NA>
4,<NA>,<NA>
5,<NA>,<NA>
6,<NA>,<NA>
7,<NA>,<NA>
8,<NA>,<NA>
9,<NA>,<NA>


In [100]:
def get_percentile(s:pd.Series, timeperiod=21)-> pd.Series:
    # store the results
    rs = pd.DataFrame(index=s.index, columns=['bull_percentile', 'bear_percentile'])

    # iterating
    for i in range(len(s)-timeperiod):
        window = s.iloc[i:i+timeperiod]
        last_value = window.iloc[-1]
        # if pd.isna(last_value):
        #     rs.at[i + timeperiod-1, 'bull_percentile'] = np.nan
        #     rs.at[i + timeperiod-1, 'bear_percentile'] = np.nan
            # continue
        masked_window = window.mask(window<0).dropna() if last_value<0 else window.mask(window>0).dropna()  
        print(masked_window)
        # calculating the percentile
        if len(masked_window) > 1:
            if last_value > 0:
                rs.iloc[i + timeperiod-1]['bull_percentile'] = (masked_window < last_value).sum() / len(masked_window)
                rs.at[i + timeperiod-1, 'bear_percentile'] = np.nan  # No bear percentile
            else:
                rs.at[i + timeperiod-1, 'bear_percentile'] = (masked_window < last_value).sum() / len(masked_window)
                rs.at[i + timeperiod-1, 'bull_percentile'] = np.nan  # No bull percentile
        else:
            rs.at[i + timeperiod-1, 'bull_percentile'] = np.nan
            rs.at[i + timeperiod-1, 'bear_percentile'] = np.nan
    return rs


In [103]:
import pandas as pd
import numpy as np

# Set the random seed for reproducibility
np.random.seed(42)

# Generate random integers between -100 and 100
random_integers = np.random.randint(-20, 101, size=1000)

# Create a Pandas Series
test_series = pd.Series(random_integers)

print(test_series)


0      82
1      31
2      72
3      -6
4      86
       ..
995    32
996    16
997    53
998    53
999    62
Length: 1000, dtype: int32


In [104]:
rs = get_percentile(test_series)


3     -6.0
7      0.0
18   -18.0
dtype: float64
1     31.0
2     72.0
4     86.0
5     51.0
6     40.0
7      0.0
8     82.0
9     62.0
10    66.0
11    54.0
12    54.0
13    67.0
14    96.0
15    79.0
16    83.0
17     3.0
19     1.0
20    32.0
dtype: float64
3     -6.0
7      0.0
18   -18.0
21   -19.0
dtype: float64
3     -6.0
7      0.0
18   -18.0
21   -19.0
dtype: float64
7      0.0
18   -18.0
21   -19.0
dtype: float64
7      0.0
18   -18.0
21   -19.0
dtype: float64
6     40.0
7      0.0
8     82.0
9     62.0
10    66.0
11    54.0
12    54.0
13    67.0
14    96.0
15    79.0
16    83.0
17     3.0
19     1.0
20    32.0
22    67.0
23    87.0
24     9.0
25    17.0
dtype: float64
7      0.0
18   -18.0
21   -19.0
26   -19.0
dtype: float64
18   -18.0
21   -19.0
26   -19.0
dtype: float64
18   -18.0
21   -19.0
26   -19.0
29     0.0
dtype: float64
18   -18.0
21   -19.0
26   -19.0
29     0.0
dtype: float64
18   -18.0
21   -19.0
26   -19.0
29     0.0
dtype: float64
18   -18.0
21   -19.0
26   -

In [105]:
rs

,bull_percentile,bear_percentile
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN
...,...,...
995,1.0,NaN
996,1.0,NaN
997,1.0,NaN
998,1.0,NaN


In [108]:
arr = [1,2,3,4,5,6,6,7,8,9,10]
df = pd.Series(arr)
df

0      1
1      2
2      3
3      4
4      5
5      6
6      6
7      7
8      8
9      9
10    10
dtype: int64

In [109]:
(df > 6).sum()

4

In [111]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Generate the test series with random integers
np.random.seed(42)
random_integers = np.random.randint(-100, 101, size=1000)
test_series = pd.Series(random_integers)

# Define a function to calculate percentiles
def get_percentile(s: pd.Series, timeperiod=21) -> pd.DataFrame:
    rs = pd.DataFrame(index=s.index, columns=['bull_percentile', 'bear_percentile'])

    for i in range(len(s) - timeperiod):
        window = s.iloc[i:i + timeperiod]
        last_value = window.iloc[-1]

        if pd.isna(last_value):
            rs.iat[i + timeperiod-1, 0] = np.nan
            rs.iat[i + timeperiod-1, 1] = np.nan
            continue

        masked_window = window.mask(window < 0).dropna() if last_value < 0 else window.mask(window > 0).dropna()

        # if len(masked_window) > 1:
        if last_value > 0:
            rs.iat[i + timeperiod-1, 0] = (masked_window < last_value).sum() / len(masked_window)
            rs.iat[i + timeperiod-1, 1] = np.nan
        else:
            rs.iat[i + timeperiod-1, 0] = (masked_window < last_value).sum() / len(masked_window)
            rs.iat[i + timeperiod-1, 1] = np.nan
        # else:
        #     rs.iat[i + timeperiod-1, 'bull_percentile'] = np.nan
        #     rs.at[i + timeperiod-1, 'bear_percentile'] = np.nan

    return rs

# Calculate the percentiles
percentiles = get_percentile(test_series)

# Create the plot
fig = go.Figure()

# Add bull_percentile trace
fig.add_trace(go.Scatter(
    x=percentiles.index,
    y=percentiles['bull_percentile'],
    mode='lines',
    name='Bull Percentile',
    line=dict(color='green')
))

# Add bear_percentile trace
fig.add_trace(go.Scatter(
    x=percentiles.index,
    y=percentiles['bear_percentile'],
    mode='lines',
    name='Bear Percentile',
    line=dict(color='red')
))

# Update layout
fig.update_layout(
    title='Bull and Bear Percentiles',
    xaxis_title='Index',
    yaxis_title='Percentile',
    showlegend=True
)

# Show the plot
fig.show()
